In [ ]:
import pandas as pd
import re

MERGED_CSV = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_with_s5p_90360_plus_match.csv"
TRIPLET_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/s5p_samples_5x5_to_32_triplet.csv"  # 你给的
OUT_MANIFEST = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest.csv"

def get_proc(path):
    if not isinstance(path, str):
        return None
    m = re.search(r"S5P_(OFFL|RPRO|NRTI)", path)
    return m.group(1) if m else None

df = pd.read_csv(MERGED_CSV, low_memory=False)

# 这些列名按你原来的 merged_file 逻辑
need_cols = ["plume_id", "lat", "lon", "plume_time",
             "S5p_path", "s5p_minus90_path", "s5p_minus360_path"]
for c in need_cols:
    if c not in df.columns:
        raise RuntimeError(f"Missing col in merged: {c}")

df["proc_0"]   = df["S5p_path"].apply(get_proc)
df["proc_90"]  = df["s5p_minus90_path"].apply(get_proc)
df["proc_360"] = df["s5p_minus360_path"].apply(get_proc)

# 只关心：t0是OFFL，但 -90或-360不是OFFL
cand = df[(df["proc_0"] == "OFFL") & ((df["proc_90"] != "OFFL") | (df["proc_360"] != "OFFL"))].copy()

# 需要重下哪个offset
def need_offsets(r):
    offs = []
    if r["proc_90"] != "OFFL":
        offs.append(90)
    if r["proc_360"] != "OFFL":
        offs.append(360)
    return ";".join(map(str, offs))

cand["need_offsets"] = cand.apply(need_offsets, axis=1)

# 可选：只保留已经出现在你 triplet 样本里的 plume（避免重下无用的）
trip = pd.read_csv(TRIPLET_CSV, low_memory=False)
if "plume_id" not in trip.columns:
    raise RuntimeError("Missing plume_id in triplet csv")
trip_plumes = set(trip["plume_id"].astype(str).unique().tolist())
cand["in_triplet"] = cand["plume_id"].astype(str).isin(trip_plumes) #为什么会输出false？

out = cand[[
    "plume_id", "lat", "lon", "plume_time",
    "S5p_path", "s5p_minus90_path", "s5p_minus360_path",
    "proc_0", "proc_90", "proc_360",
    "need_offsets", "in_triplet"
]].copy()

out.to_csv(OUT_MANIFEST, index=False)

print("Total mixed candidates:", len(cand))
print("In triplet:", int(out["in_triplet"].sum()), "/", len(out))
print("Saved manifest:", OUT_MANIFEST)
print("\nExample row:")
if len(out) > 0:
    r = out.iloc[0]
    print("plume_id:", r["plume_id"], "need_offsets:", r["need_offsets"], "in_triplet:", r["in_triplet"])
    print("t0 :", r["proc_0"], r["S5p_path"])
    print("t90:", r["proc_90"], r["s5p_minus90_path"])
    print("t360:", r["proc_360"], r["s5p_minus360_path"])

Total mixed candidates: 1675
In triplet: 0 / 1675
Saved manifest: /data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest.csv

Example row:
plume_id: GAO20191019t145209p0000-A need_offsets: 360 in_triplet: False
t0 : OFFL /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20191019T192023_20191019T210153_10448_01_010302_20191025T213811.nc
t90: OFFL /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20190720T193504_20190720T211634_09157_01_010302_20190726T213837.nc
t360: RPRO /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p_90360/S5P_RPRO_L2__CH4____20181023T184844_20181023T203014_05326_03_020400_20221115T110625.nc


In [ ]:
# download supplemental S5P OFFL data
import json
import os
import threading
import time
import zipfile
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Set

import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor

# ===== input / output =====
MANIFEST_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest.csv"
OUT_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest_with_paths.csv"

RAW_SUBDIR_NAME = "raw_data_dir_s5p_90360"
RAW_ROOT = os.path.join("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao", RAW_SUBDIR_NAME)
os.makedirs(RAW_ROOT, exist_ok=True)

# ===== CDSE =====
CDSE_USERNAME0 = "yuyaow42@gmail.com"
CDSE_PASSWORD0 = "finhah-3zihty-seHmuf"

S5P_COLLECTION_NAME = "SENTINEL-5P"
S5P_PRODUCT_TYPE = "L2__CH4___"

# 你要重下的 offset 还是 90/360
SEARCH_WINDOW_DAYS = 50   # +/-50 days around target_dt
MAX_WORKERS = 8

PLUME_COMPLETION_MARKER = "download_stub_pre.json"
BACKOFF_STATUS_CODE = 429
BACKOFF_BASE_SECONDS = 30
BACKOFF_MAX_SECONDS = 120
BACKOFF_MAX_RETRIES = 9

product_lock_map: Dict[str, threading.Lock] = {}
product_lock_map_lock = threading.Lock()


def parse_iso_datetime(value: Optional[str]) -> Optional[datetime]:
    if not isinstance(value, str) or len(value) == 0:
        return None
    normalized = value.replace("Z", "+00:00")
    try:
        return datetime.fromisoformat(normalized)
    except ValueError:
        return None


def datetime_to_query_string(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")


def datetime_to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def compute_backoff_delay(attempt: int, headers: Optional[Dict[str, str]] = None) -> int:
    retry_after = None
    if headers:
        raw_retry = headers.get("Retry-After")
        if raw_retry:
            try:
                retry_after = int(raw_retry)
            except ValueError:
                retry_after = None
    if retry_after is None:
        retry_after = BACKOFF_BASE_SECONDS * (attempt + 1)
    return min(BACKOFF_MAX_SECONDS, max(BACKOFF_BASE_SECONDS, retry_after))


def request_with_backoff(request_fn, description: str = "request"):
    for attempt in range(BACKOFF_MAX_RETRIES):
        try:
            response = request_fn()
        except requests.RequestException:
            raise
        if response.status_code == BACKOFF_STATUS_CODE:
            wait_seconds = compute_backoff_delay(attempt, response.headers)
            print(f"HTTP {BACKOFF_STATUS_CODE} on {description}; retry in {wait_seconds}s")
            response.close()
            time.sleep(wait_seconds)
            continue
        return response
    raise RuntimeError(f"Exceeded maximum retries for {description} due to repeated HTTP {BACKOFF_STATUS_CODE}")


def get_product_lock(product_name: str) -> threading.Lock:
    with product_lock_map_lock:
        lock = product_lock_map.get(product_name)
        if lock is None:
            lock = threading.Lock()
            product_lock_map[product_name] = lock
        return lock


def get_access_token(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data=data,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["access_token"]


class RefreshableAccessToken:
    def __init__(self, username: str, password: str) -> None:
        self.username = username
        self.password = password
        self.value = get_access_token(username, password)
        self.lock = threading.Lock()

    def update(self):
        with self.lock:
            self.value = get_access_token(self.username, self.password)

    def get(self) -> str:
        with self.lock:
            return self.value


def refresh_variable(variable: RefreshableAccessToken):
    while True:
        try:
            variable.update()
        except Exception:
            pass
        time.sleep(300)


def download(access_token: str, output_dir: str, product_id: str, name: str) -> Optional[str]:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, name)
    if os.path.exists(output_path):
        return output_path

    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    headers = {"Authorization": f"Bearer {access_token}"}

    session = requests.Session()
    session.headers.update(headers)

    resp = request_with_backoff(
        lambda: session.get(url, headers=headers, stream=True, timeout=300),
        description=f"download {name}",
    )

    try:
        if resp.status_code != 200:
            print(f"download failed {resp.status_code} {name}")
            return None

        with open(output_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        # 解压并只取 .nc
        if zipfile.is_zipfile(output_path):
            extract_dir = output_path + "_extracted"
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(output_path, "r") as zf:
                for member in zf.namelist():
                    if member.endswith(".nc"):
                        zf.extract(member, extract_dir)
            return extract_dir  # 你原逻辑也是接受 dir
        return output_path
    finally:
        resp.close()
        session.close()


def fetch_products(poly: str, start_ts: str, end_ts: str) -> List[Dict]:
    products: List[Dict] = []
    next_link = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
        f"$filter=Collection/Name eq '{S5P_COLLECTION_NAME}' "
        f"and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
        f"and att/OData.CSC.StringAttribute/Value eq '{S5P_PRODUCT_TYPE}') "
        f"and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON({poly})') "
        f"and ContentDate/Start gt {start_ts} "
        f"and ContentDate/Start lt {end_ts}"
        "&$top=1000"
    )

    while next_link:
        resp = request_with_backoff(lambda: requests.get(next_link, timeout=120), description="catalogue query")
        resp.raise_for_status()
        payload = resp.json()
        values = payload.get("value", [])

        for product in values:
            name = product.get("Name", "")
            # ✅ 只要 OFFL
            if "S5P_OFFL" not in name:
                continue

            start_time_str = (product.get("ContentDate") or {}).get("Start")
            acq_time = parse_iso_datetime(start_time_str) if start_time_str else None
            if acq_time is None:
                continue

            products.append({
                "Id": product.get("Id"),
                "Name": name,
                "acq_time": acq_time,
            })

        next_link = payload.get("@odata.nextLink", "") or ""
    return products


def select_closest_product(products: List[Dict], target_dt: datetime) -> Optional[Dict]:
    if not products:
        return None
    return min(products, key=lambda p: abs((p["acq_time"] - target_dt).total_seconds()))


def offset_poly(lat: float, lon: float, d: float = 0.01) -> str:
    down_left = (lon - d, lat - d)
    up_right = (lon + d, lat + d)
    return (
        "("
        + f"{down_left[0]} {down_left[1]},"
        + f"{down_left[0]} {up_right[1]},"
        + f"{up_right[0]} {up_right[1]},"
        + f"{up_right[0]} {down_left[1]},"
        + f"{down_left[0]} {down_left[1]}"
        + ")"
    )


def parse_need_offsets(s: str) -> List[int]:
    s = str(s) if s is not None else ""
    s = s.strip()
    if not s:
        return []
    out = []
    for part in s.split(";"):
        part = part.strip()
        if part:
            out.append(int(part))
    return out


def process_one(i: int, row: Dict, token: RefreshableAccessToken) -> Dict:
    plume_id = str(row["plume_id"])
    lat = float(row["lat"])
    lon = float(row["lon"])
    base_dt = parse_iso_datetime(str(row["plume_time"]))
    if base_dt is None:
        return {"index": i, "updates": {}}
    base_dt = base_dt.astimezone(timezone.utc)

    need_offsets = parse_need_offsets(row.get("need_offsets", ""))
    if not need_offsets:
        return {"index": i, "updates": {}}

    poly = offset_poly(lat, lon, d=0.01)
    updates = {}

    for off in need_offsets:
        target_dt = base_dt - timedelta(days=off)
        # ✅ 用 +/- window，允许“换一个时间”找最近 OFFL
        w0 = target_dt - timedelta(days=SEARCH_WINDOW_DAYS)
        w1 = target_dt + timedelta(days=SEARCH_WINDOW_DAYS)

        products = fetch_products(poly, datetime_to_query_string(w0), datetime_to_query_string(w1))
        if not products:
            print(f"[{plume_id}] offset {off}: no OFFL products in window")
            continue

        sel = select_closest_product(products, target_dt)
        if sel is None:
            continue

        name = sel["Name"]
        pid = sel["Id"]

        with get_product_lock(name):
            path = download(token.get(), RAW_ROOT, pid, name)
        if path is None:
            continue

        updates[off] = {
            "new_name": name,
            "new_path": path,
            "new_datetime": datetime_to_iso_z(sel["acq_time"]),
        }
        print(f"[{plume_id}] offset {off}: selected {name}")

        time.sleep(0.2)

    return {"index": i, "updates": updates}


def main():
    df = pd.read_csv(MANIFEST_CSV, low_memory=False)

    #只处理 in_triplet== 的
    if "in_triplet" in df.columns:
        df = df[df["in_triplet"] == False].copy()

    # 输出列预留
    for off in (90, 360):
        for k in ("offl_datetime", "offl_path", "offl_name"):
            col = f"s5p_minus{off}_{k}"
            if col not in df.columns:
                df[col] = ""

    token = RefreshableAccessToken(CDSE_USERNAME0, CDSE_PASSWORD0)
    t = threading.Thread(target=refresh_variable, args=(token,))
    t.daemon = True
    t.start()

    rows = df.to_dict("records")
    updates_all = [None] * len(rows)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = []
        for i, r in enumerate(rows):
            futs.append(ex.submit(process_one, i, r, token))
        for fut in futs:
            res = fut.result()
            updates_all[res["index"]] = res["updates"]

    # 写回
    for i, upd in enumerate(updates_all):
        if not upd:
            continue
        for off, rec in upd.items():
            df.at[df.index[i], f"s5p_minus{off}_offl_datetime"] = rec.get("new_datetime", "")
            df.at[df.index[i], f"s5p_minus{off}_offl_path"] = rec.get("new_path", "")
            df.at[df.index[i], f"s5p_minus{off}_offl_name"] = rec.get("new_name", "")

    df.to_csv(OUT_CSV, index=False)
    print("Saved:", OUT_CSV)
    print("RAW_ROOT:", RAW_ROOT)


if __name__ == "__main__":
    main()


[GAO20191019t145209p0000-A] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191022t150055p0000-G] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191022t151126p0000-D] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191019t191619p0000-B] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191019t174717p0000-B] offset 90: selected S5P_OFFL_L2__CH4____20190721T191559_20190721T205729_09171_01_010302_20190727T212130.nc
[GAO20191022t154327p0000-F] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191022t153329p0000-C] offset 360: selected S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc
[GAO20191019t174717p0000-B] offset 360: selected 

In [2]:
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import xarray as xr

# ======================
# Config
# ======================
IN_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest_with_centers.csv"

OUT_ROOT = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
# OUT_CSV = OUT_ROOT / "s5p_patches_3x3_to_32.csv"
OUT_CSV = OUT_ROOT / "supplement_s5p_patches_3x3_to_32.csv"

# crop 3x3 then resize -> 32x32
CROP_SIZE = 3
CROP_HALF = CROP_SIZE // 2  # 1
OUT_SIZE = 32

MAX_MISSING_RATIO = 0.20  # only enforced for t0 (pos/neg); t-90/t-360 allow NaN

CH4_CANDIDATES = [
    "methane_mixing_ratio_bias_corrected",
    "methane_mixing_ratio",
    "xch4",
]
QA_NAME = "qa_value"  # optional, currently not saved


# ======================
# Helpers
# ======================
def pick_var(ds, candidates):
    for k in candidates:
        if k in ds.variables:
            return k
    return None


def get_2d(x):
    a = np.asarray(x)
    if a.ndim == 3:
        return a[0]
    if a.ndim == 2:
        return a
    raise ValueError(f"Unexpected dims: {a.shape}")


def to_nan_invalid(da):
    """Convert _FillValue / missing_value to NaN, cast float32, return 2D."""
    a = da.values.astype(np.float32, copy=False)
    fv = da.attrs.get("_FillValue", None)
    mv = da.attrs.get("missing_value", None)
    if fv is not None:
        a = np.where(a == np.float32(fv), np.nan, a)
    if mv is not None:
        a = np.where(a == np.float32(mv), np.nan, a)
    a = np.where(np.abs(a) > 1e20, np.nan, a)
    return get_2d(a)


def missing_ratio(patch2d):
    return 1.0 - (np.isfinite(patch2d).sum() / patch2d.size)


def is_valid_path(p):
    return isinstance(p, str) and len(p) > 0 and p.lower() != "nan"


def crop_center(a2d, cy, cx, half):
    """Crop (2*half+1)x(2*half+1) around (cy,cx)."""
    H, W = a2d.shape
    y0, y1 = cy - half, cy + half + 1
    x0, x1 = cx - half, cx + half + 1
    if y0 < 0 or x0 < 0 or y1 > H or x1 > W:
        return None
    return a2d[y0:y1, x0:x1]


def nan_crop():
    return np.full((CROP_SIZE, CROP_SIZE), np.nan, dtype=np.float32)


def nan_out():
    return np.full((OUT_SIZE, OUT_SIZE), np.nan, dtype=np.float32)


def bilinear_resize_2d(src, out_h=OUT_SIZE, out_w=OUT_SIZE):
    """
    Pure numpy bilinear resize for 2D array, NaN-aware:
    - uses weighted average ignoring NaNs
    - if all contributing pixels are NaN => output NaN
    """
    src = src.astype(np.float32, copy=False)
    in_h, in_w = src.shape

    if in_h == out_h and in_w == out_w:
        return src.copy()

    # map output grid to input coords
    # align corners style
    y = np.linspace(0, in_h - 1, out_h, dtype=np.float32)
    x = np.linspace(0, in_w - 1, out_w, dtype=np.float32)
    y0 = np.floor(y).astype(np.int32)
    x0 = np.floor(x).astype(np.int32)
    y1 = np.clip(y0 + 1, 0, in_h - 1)
    x1 = np.clip(x0 + 1, 0, in_w - 1)

    wy = (y - y0).astype(np.float32)  # (out_h,)
    wx = (x - x0).astype(np.float32)  # (out_w,)

    # gather 4 neighbors with broadcasting
    Ia = src[y0[:, None], x0[None, :]]
    Ib = src[y0[:, None], x1[None, :]]
    Ic = src[y1[:, None], x0[None, :]]
    Id = src[y1[:, None], x1[None, :]]

    wa = (1 - wy)[:, None] * (1 - wx)[None, :]
    wb = (1 - wy)[:, None] * (wx)[None, :]
    wc = (wy)[:, None] * (1 - wx)[None, :]
    wd = (wy)[:, None] * (wx)[None, :]

    # NaN-aware: zero out weights where value is NaN
    mask_a = np.isfinite(Ia)
    mask_b = np.isfinite(Ib)
    mask_c = np.isfinite(Ic)
    mask_d = np.isfinite(Id)

    num = (
        np.where(mask_a, Ia * wa, 0.0) +
        np.where(mask_b, Ib * wb, 0.0) +
        np.where(mask_c, Ic * wc, 0.0) +
        np.where(mask_d, Id * wd, 0.0)
    )
    den = (
        np.where(mask_a, wa, 0.0) +
        np.where(mask_b, wb, 0.0) +
        np.where(mask_c, wc, 0.0) +
        np.where(mask_d, wd, 0.0)
    )

    out = np.where(den > 0, num / den, np.nan).astype(np.float32)
    return out


def nearest_iyix(lat, lon, lat0, lon0):
    """Nearest pixel index using equirectangular approx."""
    lat = lat.astype(np.float64, copy=False)
    lon = lon.astype(np.float64, copy=False)
    latr = np.deg2rad(lat)
    lonr = np.deg2rad(lon)
    lat0r = math.radians(float(lat0))
    lon0r = math.radians(float(lon0))

    dlon = (lonr - lon0r + np.pi) % (2*np.pi) - np.pi
    x = dlon * np.cos(0.5 * (latr + lat0r))
    y = latr - lat0r
    d2 = x * x + y * y

    flat = np.nanargmin(d2)
    iy, ix = np.unravel_index(flat, d2.shape)
    return int(iy), int(ix)


def find_neg_center(ch4_2d, pos_center, seed, avoid_radius_px=20, random_tries=80):
    """
    negative center selection:
    - corners first (must be in-bounds for crop3x3 and missing<=thr)
    - random fallback
    """
    H, W = ch4_2d.shape
    py, px = pos_center

    # For crop 3x3, valid center range is [1, H-2] and [1, W-2]
    y_min, y_max = CROP_HALF, H - CROP_HALF - 1
    x_min, x_max = CROP_HALF, W - CROP_HALF - 1

    # if the swath is too thin (rare but can happen), give up
    if y_min > y_max or x_min > x_max:
        return None

    corners = [
        (y_min, x_min),
        (y_min, x_max),
        (y_max, x_min),
        (y_max, x_max),
    ]
    corners = sorted(corners, key=lambda p: (p[0]-py)**2 + (p[1]-px)**2, reverse=True)
    for cy, cx in corners:
        p = crop_center(ch4_2d, cy, cx, CROP_HALF)
        if p is not None and missing_ratio(p) <= MAX_MISSING_RATIO:
            return cy, cx

    rng = random.Random(seed)
    for _ in range(random_tries):
        cy = rng.randint(y_min, y_max)
        cx = rng.randint(x_min, x_max)
        if (cy - py) ** 2 + (cx - px) ** 2 < avoid_radius_px ** 2:
            continue
        p = crop_center(ch4_2d, cy, cx, CROP_HALF)
        if p is not None and missing_ratio(p) <= MAX_MISSING_RATIO:
            return cy, cx

    return None


def read_ch4_2d(nc_path, ch4name):
    """Read one variable from PRODUCT group; return 2D float32 with NaNs."""
    try:
        ds = xr.open_dataset(nc_path, group="PRODUCT", engine="netcdf4", cache=False, decode_timedelta=True)
        if ch4name not in ds.variables:
            ds.close()
            return None, f"no_var:{ch4name}"
        ch4 = to_nan_invalid(ds[ch4name])
        ds.close()
        return ch4, None
    except Exception as e:
        return None, f"open_fail:{type(e).__name__}"


# ======================
# Main
# ======================
df = pd.read_csv(IN_CSV, low_memory=False)

rows = []
kept = 0
dropped = 0
err_cnt = Counter()
printed = 0

# NOTE: your csv columns are:
# ['plume_id','plume_time','lat','lon','s5p_minus360_path','s5p_minus90_path','S5p_path','proc_0','proc_90','proc_360']
# We will use exactly those.
required_cols = ["plume_id", "plume_time", "lat", "lon", "S5p_path", "s5p_minus90_offl_path", "s5p_minus360_offl_path"]
for c in required_cols:
    if c not in df.columns:
        raise RuntimeError(f"Missing column in CSV: {c}. Available: {list(df.columns)}")

for i, r in df.iterrows():
    try:
        plume_id = str(r["plume_id"])
        plume_time = str(r["plume_time"])
        lat0 = float(r["lat"])
        lon0 = float(r["lon"])

        p0 = str(r["S5p_path"])
        p90 = str(r["s5p_minus90_offl_path"])
        p360 = str(r["s5p_minus360_offl_path"])

        # ---- open t0, determine ch4 var + pos/neg centers ----
        try:
            ds0 = xr.open_dataset(p0, group="PRODUCT", engine="netcdf4", cache=False, decode_timedelta=True)
        except Exception as e:
            raise RuntimeError(f"open_t0_fail:{type(e).__name__}")

        lat = get_2d(ds0["latitude"].values)
        lon = get_2d(ds0["longitude"].values)

        ch4name = pick_var(ds0, CH4_CANDIDATES)
        if ch4name is None:
            ds0.close()
            raise RuntimeError("no_ch4")

        ch40 = to_nan_invalid(ds0[ch4name])
        ds0.close()

        cy, cx = nearest_iyix(lat, lon, lat0, lon0)

        # pos t0 crop3x3 (strict)
        pos0_small = crop_center(ch40, cy, cx, CROP_HALF)
        if pos0_small is None or missing_ratio(pos0_small) > MAX_MISSING_RATIO:
            raise RuntimeError("pos0 bad")

        # choose neg center on t0 only (strict)
        neg_center = find_neg_center(ch40, (cy, cx), seed=i)
        if neg_center is None:
            raise RuntimeError("no neg")

        neg0_small = crop_center(ch40, neg_center[0], neg_center[1], CROP_HALF)
        if neg0_small is None or missing_ratio(neg0_small) > MAX_MISSING_RATIO:
            raise RuntimeError("neg0 bad")

        # resize to 32x32
        pos0 = bilinear_resize_2d(pos0_small, OUT_SIZE, OUT_SIZE)
        neg0 = bilinear_resize_2d(neg0_small, OUT_SIZE, OUT_SIZE)

        # ---- t-90 / t-360 : allow missing => NaN outputs if file missing or open fails or oob ----
        has90 = is_valid_path(p90) and Path(p90).exists()
        has360 = is_valid_path(p360) and Path(p360).exists()

        # default NaNs
        pos90 = nan_out()
        neg90 = nan_out()
        pos360 = nan_out()
        neg360 = nan_out()

        if has90:
            ch490, e = read_ch4_2d(p90, ch4name)
            if e is None and ch490 is not None:
                p = crop_center(ch490, cy, cx, CROP_HALF)
                n = crop_center(ch490, neg_center[0], neg_center[1], CROP_HALF)
                if p is not None:
                    pos90 = bilinear_resize_2d(p, OUT_SIZE, OUT_SIZE)
                if n is not None:
                    neg90 = bilinear_resize_2d(n, OUT_SIZE, OUT_SIZE)
            else:
                has90 = False  # mark as unusable

        if has360:
            ch4360, e = read_ch4_2d(p360, ch4name)
            if e is None and ch4360 is not None:
                p = crop_center(ch4360, cy, cx, CROP_HALF)
                n = crop_center(ch4360, neg_center[0], neg_center[1], CROP_HALF)
                if p is not None:
                    pos360 = bilinear_resize_2d(p, OUT_SIZE, OUT_SIZE)
                if n is not None:
                    neg360 = bilinear_resize_2d(n, OUT_SIZE, OUT_SIZE)
            else:
                has360 = False

        # stack shape: (T=3, 32, 32)
        pos = np.stack([pos0, pos90, pos360], axis=0).astype(np.float32)
        neg = np.stack([neg0, neg90, neg360], axis=0).astype(np.float32)

        # save
        pos_path = OUT_ROOT / "patches" / "pos" / plume_id / "s5p_3x3_to_32.npz"
        neg_path = OUT_ROOT / "patches" / "neg" / plume_id / "s5p_3x3_to_32.npz"
        pos_path.parent.mkdir(parents=True, exist_ok=True)
        neg_path.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(pos_path, ch4=pos, meta={"label": 1, "plume_id": plume_id, "ch4_var": ch4name})
        np.savez_compressed(neg_path, ch4=neg, meta={"label": 0, "plume_id": plume_id, "ch4_var": ch4name})

        rows.append({
            "plume_id": plume_id,
            "plume_time": plume_time,
            "lat": lat0,
            "lon": lon0,
            "S5p_path": p0,
            "s5p_minus90_path": p90,
            "s5p_minus360_path": p360,
            "has_90": bool(has90),
            "has_360": bool(has360),
            "ch4_var": ch4name,

            "pos_npz_path": str(pos_path),
            "neg_npz_path": str(neg_path),

            "pos_center_iy": int(cy),
            "pos_center_ix": int(cx),
            "neg_center_iy": int(neg_center[0]),
            "neg_center_ix": int(neg_center[1]),

            # missing ratios computed on SMALL crops (more meaningful)
            "pos_missing_small_t0": float(missing_ratio(pos0_small)),
            "neg_missing_small_t0": float(missing_ratio(neg0_small)),
            "pos_missing_small_t-90": float(missing_ratio(crop_center(read_ch4_2d(p90, ch4name)[0], cy, cx, CROP_HALF)) if (has90 and Path(p90).exists() and read_ch4_2d(p90, ch4name)[0] is not None and crop_center(read_ch4_2d(p90, ch4name)[0], cy, cx, CROP_HALF) is not None) else np.nan),
            "pos_missing_small_t-360": float(missing_ratio(crop_center(read_ch4_2d(p360, ch4name)[0], cy, cx, CROP_HALF)) if (has360 and Path(p360).exists() and read_ch4_2d(p360, ch4name)[0] is not None and crop_center(read_ch4_2d(p360, ch4name)[0], cy, cx, CROP_HALF) is not None) else np.nan),
        })

        kept += 1
        if kept % 200 == 0:
            print(f"kept {kept} / processed {i+1} (dropped {dropped})")

    except Exception as e:
        dropped += 1
        err_cnt[str(e)] += 1
        if printed < 5:
            print("\n---- FAIL EXAMPLE ----")
            print("row i =", i)
            print("error =", repr(e))
            printed += 1
        continue

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("Top errors:", err_cnt.most_common(10))
print("Saved:", OUT_CSV)
print("Kept:", kept, "Dropped:", dropped, "Total:", len(df))
print("NPZ ch4 array shape per sample = (3, 32, 32)")


getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20191019T192023_20191019T210153_10448_01_010302_20191025T213811.nc: Operation not supported



---- FAIL EXAMPLE ----
row i = 0
error = RuntimeError('no neg')


getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20191019T192023_20191019T210153_10448_01_010302_20191025T213811.nc: Operation not supported



---- FAIL EXAMPLE ----
row i = 1
error = RuntimeError('pos0 bad')


getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20191019T192023_20191019T210153_10448_01_010302_20191025T213811.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc: Operation not supported
/tmp/ipykernel_1590680/636345246.py:147: RuntimeWarning: invalid value encountered in divide
  out = np.where(den > 0, num / den, np.nan).astype(np.float32)
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20181128T191103_20181128T205232_05837_01_010202_20181205T131651.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Meth

...

kept 1000 / processed 1375 (dropped 375)


getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20230304T174126_20230304T192256_27926_03_020400_20230306T095551.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220310T181254_20220310T195425_22833_02_020301_20220312T102036.nc: Operation not supported
/tmp/ipykernel_1590680/636345246.py:147: RuntimeWarning: invalid value encountered in divide
  out = np.where(den > 0, num / den, np.nan).astype(np.float32)
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220310T181254_20220310T195425_22833_02_020301_20220312T102036.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220310T181254_20220310T195425_22833_02_020301_20220312T102036.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Meth

kept 1200 / processed 1589 (dropped 389)


getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s5p/S5P_OFFL_L2__CH4____20230626T120826_20230626T134956_29540_03_020500_20230628T042345.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220701T111459_20220701T125629_24432_02_020301_20220703T032810.nc: Operation not supported
/tmp/ipykernel_1590680/636345246.py:147: RuntimeWarning: invalid value encountered in divide
  out = np.where(den > 0, num / den, np.nan).astype(np.float32)
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220701T111459_20220701T125629_24432_02_020301_20220703T032810.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_s5p_90360/S5P_OFFL_L2__CH4____20220701T111459_20220701T125629_24432_02_020301_20220703T032810.nc: Operation not supported
getfattr: /mnt/engg-leung/Research_No9_Meth

Top errors: [('pos0 bad', 244), ('no neg', 159)]
Saved: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/supplement_s5p_patches_3x3_to_32.csv
Kept: 1272 Dropped: 403 Total: 1675
NPZ ch4 array shape per sample = (3, 32, 32)


In [10]:
ds1=pd.read_csv('/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/supplement_s5p_patches_3x3_to_32.csv')
ds2=pd.read_csv('/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/s5p_patches_3x3_to_32.csv')

cols1 = set(ds1.columns)
cols2 = set(ds2.columns)
only_in_ds1 = sorted(cols1 - cols2)
only_in_ds2 = sorted(cols2 - cols1)
print("Only in ds1:", only_in_ds1)
print("Only in ds2:", only_in_ds2)

ds=pd.concat([ds1,ds2],axis=0,ignore_index=True)
print(len(ds1))
print(len(ds2))
print(len(ds))
ds.to_csv('/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/full_s5p_patches_3x3_to_32.csv')

Only in ds1: []
Only in ds2: []
1272
3503
4775


In [29]:
# 划分训练集和测试集
import pandas as pd
all_csv = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/full_s5p_patches_3x3_to_32.csv'
df = pd.read_csv(all_csv)
print(len(df))
df['date'] = pd.to_datetime(df['plume_time'])
# train_df = df[(df['date'] < '2024-10-10') | (df['date'] >= '2025-10-01')]  # 2025-04-01
# test_df = df[(df['date'] >= '2024-10-10') & (df['date'] < '2025-01-01')]
train_df = df[(df['date'] < '2025-6-10')]  # 2025-04-01
test_df = df[(df['date'] >= '2025-6-10')]
print(f'training set {len(train_df)} testing set {len(test_df)} train ratio {len(train_df)/(len(train_df)+len(test_df)):.3f}')

train_csv = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/train_2025_full.csv'
train_df.to_csv(train_csv, index=False)
print(f'train set size {len(train_df)}')

test_csv = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/test_2025_full.csv'
test_df.to_csv(test_csv, index=False)
print(f'test set size {len(test_df)}')


4775
training set 3852 testing set 923 train ratio 0.807
train set size 3852
test set size 923


In [30]:
# find center positions and labeling 
import pandas as pd
from pathlib import Path

# ===== input =====
TRAIN_IN = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/train_2025_full.csv"
TEST_IN  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/test_2025_full.csv"

OUT_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl")
TRAIN_OUT = OUT_DIR / "2025_train.csv"
TEST_OUT  = OUT_DIR / "2025_test.csv"


KEEP_BASE = [
    "plume_id",
    "plume_time",
    "lat",
    "lon",
    "has_90",
    "has_360",
    "ch4_var",
]

CENTER_COLS = [
    "pos_center_iy",
    "pos_center_ix",
    "neg_center_iy",
    "neg_center_ix",
]


def explode_csv(path_in, path_out):
    df = pd.read_csv(path_in)

    rows = []

    for _, r in df.iterrows():
        base = {k: r[k] for k in KEEP_BASE}

        # ---------- positive ----------
        rows.append({
            **base,
            "image_path": r["pos_npz_path"],
            "center_iy": r["pos_center_iy"],
            "center_ix": r["pos_center_ix"],
            "label": 1,
        })

        # ---------- negative ----------
        rows.append({
            **base,
            "image_path": r["neg_npz_path"],
            "center_iy": r["neg_center_iy"],
            "center_ix": r["neg_center_ix"],
            "label": 0,
        })

    out = pd.DataFrame(rows)

    # keep strict column order
    out = out[
        [
            "plume_id",
            "plume_time",
            "lat",
            "lon",
            "has_90",
            "has_360",
            "ch4_var",
            "image_path",
            "center_iy",
            "center_ix",
            "label",
        ]
    ]

    out.to_csv(path_out, index=False)
    print("Saved:", path_out, "rows:", len(out))


explode_csv(TRAIN_IN, TRAIN_OUT)
explode_csv(TEST_IN, TEST_OUT)

Saved: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/2025_train.csv rows: 7704
Saved: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_3x3_to_32_offl/2025_test.csv rows: 1846


In [ ]:
# crop to 5*5 to 32x32 triplet samples
import os
import random
import warnings
from pathlib import Path
from collections import Counter
from contextlib import contextmanager
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import cv2
from netCDF4 import Dataset


# ============== input ==============
IN_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s5p/redownload_offl_90360_manifest_with_centers.csv"

OUT_ROOT = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s5p_patches_5x5_to_32_offl_triplet")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_CSV = "./supplement_s5p_samples_5x5_to_32_triplet.csv"

POS_DIR = OUT_ROOT / "samples2" / "pos"
NEG_DIR = OUT_ROOT / "samples2" / "neg"
POS_DIR.mkdir(parents=True, exist_ok=True)
NEG_DIR.mkdir(parents=True, exist_ok=True)

# ============== crop config ==============
CROP_SIZE = 5
CROP_HALF = CROP_SIZE // 2
OUT_SIZE = 32

MAX_MISSING_RATIO_T0 = 0.50

NEG_EXCLUDE_HALF = 5           # outside 11x11
NEG_RANDOM_TRIES = 50          # ✅ 按你说的：随机 50 次
# corners are tried first

# parallel
MAX_WORKERS = 8                # 网络盘建议 4~8
PRINT_EVERY = 50

CH4_CANDIDATES = [
    "methane_mixing_ratio_bias_corrected",
    "methane_mixing_ratio",
    "xch4",
]

warnings.filterwarnings("ignore", category=RuntimeWarning)


@contextmanager
def silence_fd2():
    """Kill HDF5/getfattr spam."""
    devnull = os.open(os.devnull, os.O_WRONLY)
    old = os.dup(2)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(old, 2)
        os.close(devnull)
        os.close(old)


def get_2d(a):
    a = np.asarray(a)
    if a.ndim == 3:
        return a[0]
    if a.ndim == 2:
        return a
    raise ValueError(f"Unexpected dims: {a.shape}")


def to_nan_invalid(arr, attrs=None):
    a = np.array(arr, dtype=np.float32, copy=False)
    attrs = attrs or {}
    fv = attrs.get("_FillValue", None)
    mv = attrs.get("missing_value", None)
    if fv is not None:
        a = np.where(a == np.float32(fv), np.nan, a)
    if mv is not None:
        a = np.where(a == np.float32(mv), np.nan, a)
    a = np.where(np.abs(a) > 1e20, np.nan, a)
    return get_2d(a)


def crop_center(a2d, cy, cx, half):
    H, W = a2d.shape
    y0, y1 = cy - half, cy + half + 1
    x0, x1 = cx - half, cx + half + 1
    if y0 < 0 or x0 < 0 or y1 > H or x1 > W:
        return None
    return a2d[y0:y1, x0:x1]


def missing_ratio(patch2d):
    return 1.0 - (np.isfinite(patch2d).sum() / patch2d.size)


def nan_out():
    return np.full((OUT_SIZE, OUT_SIZE), np.nan, dtype=np.float32)


def resize_nan_aware(src2d):
    src = src2d.astype(np.float32, copy=False)
    fin = np.isfinite(src)
    v = np.where(fin, src, 0.0).astype(np.float32)
    w = fin.astype(np.float32)

    v_r = cv2.resize(v, (OUT_SIZE, OUT_SIZE), interpolation=cv2.INTER_LINEAR)
    w_r = cv2.resize(w, (OUT_SIZE, OUT_SIZE), interpolation=cv2.INTER_LINEAR)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(w_r > 1e-6, v_r / w_r, np.nan).astype(np.float32)


def parse_centers(s):
    # "cy,cx;cy,cx" -> [(cy,cx),...]
    s = str(s) if s is not None else ""
    s = s.strip()
    if not s:
        return []
    out = []
    for part in s.split(";"):
        part = part.strip()
        if not part:
            continue
        cy, cx = part.split(",")
        out.append((int(float(cy)), int(float(cx))))
    return out


def pick_ch4_var(prod):
    for k in CH4_CANDIDATES:
        if k in prod.variables:
            return k
    return None


def read_ch4(path_nc, ch4name_hint=None):
    """
    Read PRODUCT/ch4 -> 2D float32 with NaNs
    returns (ch4_2d or None, ok_bool, ch4name_used)
    """
    if not path_nc or str(path_nc).lower() == "nan":
        return None, False, ch4name_hint

    p = Path(str(path_nc))
    if not p.exists():
        return None, False, ch4name_hint

    try:
        with silence_fd2():
            ds = Dataset(str(p), "r")
        prod = ds.groups["PRODUCT"]

        ch4name = ch4name_hint if (ch4name_hint and ch4name_hint in prod.variables) else pick_ch4_var(prod)
        if ch4name is None:
            ds.close()
            return None, False, ch4name_hint

        v = prod.variables[ch4name]
        a = to_nan_invalid(v[:], getattr(v, "__dict__", {}))
        ds.close()
        return a, True, ch4name
    except Exception:
        return None, False, ch4name_hint


def outside_exclude_11x11(py, px, cy, cx):
    return not (abs(cy - py) <= NEG_EXCLUDE_HALF and abs(cx - px) <= NEG_EXCLUDE_HALF)


def find_neg_center(ch4_t0, py, px, seed):
    """
    Neg selection on t0 only:
    1) try 4 corners (in-bounds and outside 11x11)
    2) random try 50 times; first valid -> stop
    validity: crop exists and missing_ratio<=0.5 on t0
    """
    H, W = ch4_t0.shape
    y_min, y_max = CROP_HALF, H - CROP_HALF - 1
    x_min, x_max = CROP_HALF, W - CROP_HALF - 1
    if y_min > y_max or x_min > x_max:
        return None

    corners = [(y_min, x_min), (y_min, x_max), (y_max, x_min), (y_max, x_max)]
    # far first
    corners.sort(key=lambda c: (c[0]-py)**2 + (c[1]-px)**2, reverse=True)

    for cy, cx in corners:
        if not outside_exclude_11x11(py, px, cy, cx):
            continue
        p = crop_center(ch4_t0, cy, cx, CROP_HALF)
        if p is not None and missing_ratio(p) <= MAX_MISSING_RATIO_T0:
            return cy, cx

    rng = random.Random(seed)
    for _ in range(NEG_RANDOM_TRIES):
        cy = rng.randint(y_min, y_max)
        cx = rng.randint(x_min, x_max)
        if not outside_exclude_11x11(py, px, cy, cx):
            continue
        p = crop_center(ch4_t0, cy, cx, CROP_HALF)
        if p is not None and missing_ratio(p) <= MAX_MISSING_RATIO_T0:
            return cy, cx

    return None


def process_one(args):
    idx, row, pos_base, neg_base = args

    plume_id = str(row["plume_id"])
    plume_time = str(row["plume_time"])
    lat0 = float(row["lat"])
    lon0 = float(row["lon"])

    p0 = str(row["S5p_path"])
    p90 = row.get("s5p_minus90_offl_path", "") or row["s5p_minus90_path"]
    p360 = row.get("s5p_minus360_offl_path", "") or row["s5p_minus360_path"]

    py = int(float(row["nearest_iy"]))
    px = int(float(row["nearest_ix"]))
    pos_centers = parse_centers(row.get("pos_centers", ""))

    # must have at least one pos center
    if not pos_centers:
        raise RuntimeError("no_pos_centers_precomputed")

    # Read 3 arrays (each at most once)
    ch4_t0, ok0, ch4name_used = read_ch4(p0, row.get("ch4_var", None))
    if not ok0 or ch4_t0 is None:
        raise RuntimeError("open_t0_fail")

    ch4_90, ok90, _ = read_ch4(p90, ch4name_used)
    ch4_360, ok360, _ = read_ch4(p360, ch4name_used)

    H, W = ch4_t0.shape

    out_rows = []
    pos_dir = Path(pos_base) / plume_id
    neg_dir = Path(neg_base) / plume_id
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    # for each pos, save pos; try neg
    for j, (cy, cx) in enumerate(pos_centers):
        pos0_small = crop_center(ch4_t0, cy, cx, CROP_HALF)
        if pos0_small is None or missing_ratio(pos0_small) > MAX_MISSING_RATIO_T0:
            continue

        pos0 = resize_nan_aware(pos0_small)

        if ok90 and ch4_90 is not None:
            p = crop_center(ch4_90, cy, cx, CROP_HALF)
            pos90 = resize_nan_aware(p) if p is not None else nan_out()
            has90 = True
        else:
            pos90 = nan_out()
            has90 = False

        if ok360 and ch4_360 is not None:
            p = crop_center(ch4_360, cy, cx, CROP_HALF)
            pos360 = resize_nan_aware(p) if p is not None else nan_out()
            has360 = True
        else:
            pos360 = nan_out()
            has360 = False

        pos_stack = np.stack([pos0, pos90, pos360], axis=0).astype(np.float32)
        pos_npz = pos_dir / f"s5p_pos_{j:02d}.npz"
        np.savez_compressed(
            pos_npz,
            ch4=pos_stack,
            meta={"label": 1, "plume_id": plume_id, "ch4_var": ch4name_used,
                  "center_iy": int(cy), "center_ix": int(cx),
                  "nearest_iy": int(py), "nearest_ix": int(px),
                  "has_90": bool(has90), "has_360": bool(has360)}
        )

        out_rows.append({
            "plume_id": plume_id, "plume_time": plume_time, "lat": lat0, "lon": lon0,
            "has_90": bool(has90), "has_360": bool(has360), "ch4_var": ch4name_used,
            "image_path": str(pos_npz), "center_iy": int(cy), "center_ix": int(cx), "label": 1
        })

        # neg on t0 only; if fail -> keep only pos
        neg_center = find_neg_center(ch4_t0, py, px, seed=idx * 1000 + j)
        if neg_center is None:
            continue
        ncy, ncx = neg_center
        neg0_small = crop_center(ch4_t0, ncy, ncx, CROP_HALF)
        if neg0_small is None or missing_ratio(neg0_small) > MAX_MISSING_RATIO_T0:
            continue

        neg0 = resize_nan_aware(neg0_small)

        if ok90 and ch4_90 is not None:
            p = crop_center(ch4_90, ncy, ncx, CROP_HALF)
            neg90 = resize_nan_aware(p) if p is not None else nan_out()
        else:
            neg90 = nan_out()

        if ok360 and ch4_360 is not None:
            p = crop_center(ch4_360, ncy, ncx, CROP_HALF)
            neg360 = resize_nan_aware(p) if p is not None else nan_out()
        else:
            neg360 = nan_out()

        neg_stack = np.stack([neg0, neg90, neg360], axis=0).astype(np.float32)
        neg_npz = neg_dir / f"s5p_neg_{j:02d}.npz"
        np.savez_compressed(
            neg_npz,
            ch4=neg_stack,
            meta={"label": 0, "plume_id": plume_id, "ch4_var": ch4name_used,
                  "center_iy": int(ncy), "center_ix": int(ncx),
                  "nearest_iy": int(py), "nearest_ix": int(px),
                  "has_90": bool(has90), "has_360": bool(has360)}
        )

        out_rows.append({
            "plume_id": plume_id, "plume_time": plume_time, "lat": lat0, "lon": lon0,
            "has_90": bool(has90), "has_360": bool(has360), "ch4_var": ch4name_used,
            "image_path": str(neg_npz), "center_iy": int(ncy), "center_ix": int(ncx), "label": 0
        })

    return out_rows


def main():
    df = pd.read_csv(IN_CSV, low_memory=False)

    req = ["plume_id","plume_time","lat","lon","S5p_path","s5p_minus90_path","s5p_minus360_path",
           "nearest_iy","nearest_ix","pos_centers"]
    for c in req:
        if c not in df.columns:
            raise RuntimeError(f"Missing col {c}")

    rows = df.to_dict("records")
    total = len(rows)

    tasks = [(i, rows[i], str(POS_DIR), str(NEG_DIR)) for i in range(total)]

    all_samples = []
    err_cnt = Counter()
    done = 0

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(process_one, t) for t in tasks]
        for fut in as_completed(futs):
            done += 1
            try:
                all_samples.extend(fut.result())
            except Exception as e:
                err_cnt[str(e)] += 1

            if (done % PRINT_EVERY) == 0 or done == total:
                pct = 100.0 * done / total
                print(f"[{done}/{total} | {pct:5.1f}%] samples={len(all_samples)} errors={sum(err_cnt.values())}")

    out = pd.DataFrame(all_samples)
    out = out[["plume_id","plume_time","lat","lon","has_90","has_360","ch4_var","image_path","center_iy","center_ix","label"]]
    out.to_csv(OUT_CSV, index=False)

    print("Saved:", OUT_CSV)
    print("Total plumes:", total, "Total samples:", len(out))
    print("Top errors:", err_cnt.most_common(10))


if __name__ == "__main__":
    main()


[50/1675 |   3.0%] samples=747 errors=0
[100/1675 |   6.0%] samples=1443 errors=0
[150/1675 |   9.0%] samples=2129 errors=1
[200/1675 |  11.9%] samples=2825 errors=1
[250/1675 |  14.9%] samples=3575 errors=1
